In [ ]:
# [NB01-C01] SETUP
# What: install pm4py, import all the libraries used in this notebook and
#       create the output folder for the figures.
# Why: pm4py is not pre-installed on Google Colab; the version is pinned so
#      that Colab and the local environment behave the same way; the figures
#      folder must be created here or plt.savefig would fail on a clean runtime.
# Expected: no errors (on Colab the pm4py install takes about one minute).

%pip install pm4py==2.7.23.1 -q

import os                         # filesystem helpers (folder creation, file checks)
import pandas as pd               # dataframes and CSV handling
import numpy as np                # numeric helpers
import matplotlib.pyplot as plt   # plots
import seaborn as sns             # nicer statistical plots
import pm4py                      # process mining library used in the course
from scipy.stats import shapiro   # normality test for the duration distribution

# Create the folder where the report figures will be saved (safe if it exists)
os.makedirs('figures', exist_ok=True)

In [ ]:
# [NB01-C02] LOAD THE DATASET
# What: read the exam event log from CSV into a pandas dataframe.
# Why: pm4py uses dataframes as the standard format for event logs.
# Expected: 25,115 events (rows) and 27 columns.

# Path to the exam dataset (on Google Colab: upload the file and change this path)
filepath = '../dataset_for_exam.csv'

# Defensive check: stop with a clear message if the file is not where we expect
if not os.path.exists(filepath):
    raise FileNotFoundError('Dataset not found at ' + filepath +
                            ' - upload dataset_for_exam.csv and update the filepath variable.')

# Read the CSV file into a dataframe
event_logs = pd.read_csv(filepath, sep=',')

# Print the size of the log to confirm the load worked
print('Number of rows (events): {}'.format(len(event_logs)))
print('Number of columns: {}'.format(len(event_logs.columns)))

# Show the dataframe for a first visual inspection
event_logs

In [ ]:
# [NB01-C03] FIRST LOOK AT COLUMNS AND TYPES
# What: list every column with its dtype and number of non-null values.
# Why: before touching the data we need to know which attributes exist and
#      how sparse they are (many clinical columns are filled only on the
#      event they belong to, e.g. vital signs only on "Vital sign check").
# Expected: only stay_id, time and activity are always filled; the clinical
#      and medication columns have many missing values by design.

# Print column name, dtype and non-null count for each column
event_logs.info()

In [ ]:
# [NB01-C04] RENAME TO THE PM4PY STANDARD AND SORT THE LOG
# What: rename the three core columns to the PM4Py/XES naming standard,
#       convert the timestamp to datetime, and sort events inside each case.
# Why: every pm4py function expects case:concept:name / concept:name /
#      time:timestamp; sorting by case and time is required for any
#      control-flow analysis. We also keep the original row number as a
#      tie-breaker because many events share the exact same timestamp.
# Expected: same number of rows, three renamed columns, log ordered by case.

# Comply to the naming standard of PM4PY
event_logs.rename(columns={'stay_id': 'case:concept:name',
                           'activity': 'concept:name',
                           'time': 'time:timestamp'}, inplace=True)

# Convert the timestamp column from string to datetime
event_logs['time:timestamp'] = pd.to_datetime(event_logs['time:timestamp'], errors='coerce')

# pm4py treats case ids as strings, so we cast them once here to avoid
# type mismatches later (int vs str) when filtering cases
event_logs['case:concept:name'] = event_logs['case:concept:name'].astype(str)

# Keep the original row number: it preserves the recording order when two
# events of the same case have the same timestamp
event_logs['source_row'] = np.arange(len(event_logs))

# Sort by case, then time, then original row order (stable tie-breaker)
event_logs = event_logs.sort_values(['case:concept:name', 'time:timestamp', 'source_row']).reset_index(drop=True)

# Verify: no timestamp failed the datetime conversion
print('Timestamps that failed conversion: {}'.format(event_logs['time:timestamp'].isna().sum()))

# Verify the calendar range of the log: the dates are in the future because
# the source is de-identified with a date shift (privacy protection).
# Durations and the order of events remain valid; calendar dates must never
# be interpreted as real-world dates.
print('First timestamp in the log: {}'.format(event_logs['time:timestamp'].min()))
print('Last timestamp in the log: {}'.format(event_logs['time:timestamp'].max()))

event_logs

In [ ]:
# [NB01-C05] ACTIVITY FREQUENCIES
# What: count how many times each activity appears in the log and plot it.
# Why: with only 6 distinct activities, the frequency profile tells us which
#      steps are administrative (once per stay) and which are clinical
#      recordings that can repeat many times inside the same stay.
# Expected: "Enter the ED" and "Triage in the ED" exactly 1,820 (once per
#      stay); medication and vital-sign activities much higher; if any other
#      activity exceeds 1,820 we must find out why before going on.

# Compute frequency of activities
activity_counts = event_logs['concept:name'].value_counts()
print('Frequency of Activities:\n', activity_counts)

# "Discharge from the ED" appears more than once per stay: hypothesis to
# verify -> the log stores one discharge row per diagnosis code.
# First check: do the totals match?
print('\nDischarge events: {}'.format((event_logs['concept:name'] == 'Discharge from the ED').sum()))
print('Non-null diagnosis rows: {}'.format(event_logs['diagnosis_code'].notna().sum()))

# Helper: count the non-null values of a column inside one case
def count_non_null(values):
    return values.dropna().shape[0]

# Second, stronger check: the hypothesis must hold case by case, not only on
# the totals. For every stay: discharge rows == max(number of diagnosis rows, 1)
discharge_per_stay = event_logs[event_logs['concept:name'] == 'Discharge from the ED'].groupby('case:concept:name').size()
diagnosis_per_stay = event_logs.groupby('case:concept:name')['diagnosis_code'].agg(count_non_null)
expected_discharges = diagnosis_per_stay.clip(lower=1)
mismatching_stays = (discharge_per_stay.reindex(expected_discharges.index, fill_value=0) != expected_discharges).sum()
print('Stays where discharge rows != max(diagnosis rows, 1): {}'.format(mismatching_stays))

# Transform the Series into a dataframe for plotting
df_plot = activity_counts.reset_index()
df_plot.columns = ['Activity', 'Count']

# Create the bar plot
plt.figure(figsize=(10, 5))
sns.barplot(x='Activity', y='Count', data=df_plot, color='#5b9bd5')

# Rotate x-axis labels so they stay readable
plt.xticks(rotation=30)
plt.title('Frequency of Activities')
plt.ylabel('Count')
plt.tight_layout()

# Save the figure for the report, then show it
plt.savefig('figures/NB01_activity_frequencies.png', dpi=200)
plt.show()

# Interpretation: "Enter the ED" and "Triage in the ED" appear exactly once
# per stay (1,820), while the clinical activities repeat several times per
# stay. The discharge hypothesis is verified case by case with zero
# exceptions: every stay has exactly one discharge row per diagnosis code
# (or one row when no diagnosis is recorded). It is a data-representation
# choice, not a repeated activity - and it is exactly why the "abstract
# view" at the end of this notebook is needed before control-flow analysis.

In [ ]:
# [NB01-C06] START AND END ACTIVITIES
# What: retrieve the first and last activity of every case with pm4py.
# Why: the scenario states that every stay starts with "Enter the ED" and
#      ends with "Discharge from the ED". We verify this instead of
#      assuming it: it is the standard sanity check used after every
#      loading/filtering step in the course.
# Expected: a single start activity and a single end activity, both with
#      count 1,820. Any other value would reveal incomplete cases.

# Get the start activities of all cases
start_activities = pm4py.get_start_activities(event_logs)

# Get the end activities of all cases
end_activities = pm4py.get_end_activities(event_logs)

print('Start activities: {}'.format(start_activities))
print('End activities: {}'.format(end_activities))

# Interpretation: both dictionaries contain exactly one key with value
# 1,820, so the log has no truncated cases at the boundaries and no case
# needs to be dropped for incompleteness.

In [ ]:
# [NB01-C07] EVENTS PER CASE
# What: compute how many events each stay contains and describe the
#       distribution (mean, quartiles, min, max) with a histogram.
# Why: stays with very many events signal repeated clinical recordings;
#      knowing the spread helps to spot anomalous cases without deleting
#      them blindly.
# Expected: median around 10-15 events per stay, with a long right tail.

# Count the number of events of each case
events_per_case = event_logs.groupby('case:concept:name').size()

# Describe the distribution of events per case
print(events_per_case.describe())

# Plot the histogram of events per case
plt.figure(figsize=(10, 5))
plt.hist(events_per_case, bins=40, edgecolor='black', alpha=0.7)
plt.xlabel('Events per stay')
plt.ylabel('Number of stays')
plt.title('Histogram of events per stay')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

# Save the figure for the report, then show it
plt.savefig('figures/NB01_events_per_case.png', dpi=200)
plt.show()

# Interpretation: the distribution is right-skewed. The stays in the tail
# are kept and flagged (not removed): a long stay with many vital-sign
# checks can be a legitimate severe patient, not noise.

In [ ]:
# [NB01-C08] MISSING VALUES AUDIT
# What: compute the percentage of missing values for every column.
# Why: the clinical attributes are only filled on the event they refer to
#      (e.g. temperature only on "Triage in the ED" / "Vital sign check").
#      We must quantify this instead of imputing values: clinical imputation
#      would fabricate data.
# Expected: near 0% missing on the three core columns; high percentages on
#      the clinical and medication columns, from about 50% (drug columns,
#      filled on all medication events) up to 99% (rhythm).

# Compute the percentage of missing values per column
missing_pct = (event_logs.isna().mean() * 100).round(2)

# Transform the Series into a dataframe for a readable table
missing_df = missing_pct.reset_index()
missing_df.columns = ['column', 'missing_pct']

# Sort from the most to the least complete column
missing_df = missing_df.sort_values('missing_pct').reset_index(drop=True)

missing_df

# Interpretation: missingness here is structural (attributes belong to
# specific event types), not a data-quality failure. The correct handling
# is to separate case-level attributes from event-level attributes (done
# in the case table below), not to impute.

In [ ]:
# [NB01-C09] DUPLICATES AND SAME-TIMESTAMP EVENTS
# What: count fully duplicated rows and rows sharing the same
#       (case, timestamp, activity) triple.
# Why: hypothesis to verify, not assume: repeated (case, time, activity)
#      rows are legitimate clinical recordings (e.g. several drugs dispensed
#      at the same minute), NOT logging errors. If true, they must be kept
#      in the raw log and only aggregated in the abstract view.
# Expected: zero fully duplicated rows; several thousand repeated triples.

# Count rows that are completely identical on every column except source_row
full_duplicates = event_logs.drop(columns=['source_row']).duplicated().sum()
print('Fully duplicated rows: {}'.format(full_duplicates))

# Two ways to count the repeated triples, reported separately to be precise:
# (a) surplus rows = the extra recordings beyond the first one of each triple
surplus_rows = event_logs.duplicated(subset=['case:concept:name', 'time:timestamp', 'concept:name']).sum()
print('Surplus rows beyond the first of each (case, timestamp, activity) triple: {}'.format(surplus_rows))

# (b) involved rows = all the rows that belong to a repeated triple
involved_rows = event_logs.duplicated(subset=['case:concept:name', 'time:timestamp', 'concept:name'], keep=False).sum()
print('Total rows involved in repeated triples: {}'.format(involved_rows))

# Look at one concrete example to verify the hypothesis: we pick the first
# repeated triple and show its rows
example_mask = event_logs.duplicated(subset=['case:concept:name', 'time:timestamp', 'concept:name'], keep=False)
example_key = event_logs[example_mask].iloc[0][['case:concept:name', 'time:timestamp', 'concept:name']]
example_rows = event_logs[(event_logs['case:concept:name'] == example_key['case:concept:name']) &
                          (event_logs['time:timestamp'] == example_key['time:timestamp']) &
                          (event_logs['concept:name'] == example_key['concept:name'])]
example_rows

# Interpretation: no full duplicates exist; 12,268 rows belong to repeated
# triples, of which 8,289 are surplus recordings beyond the first one. The
# example shows different drug values on each row, so the repeated triples
# carry real information (different drugs at the same minute). We keep them
# all in the raw log; the abstract view in [NB01-C16] will aggregate the
# 8,289 surplus rows for control-flow analysis only.

In [ ]:
# [NB01-C10] PHYSIOLOGICAL VALUES: MEASUREMENT AUDIT AND UNIT ERRORS
# What: for each physiological measure, count the really recorded values
#       (before any numeric conversion), convert to numeric, and test the
#       hypothesis that out-of-range temperatures are unit errors (Celsius
#       instead of Fahrenheit) before discarding anything.
# Why: pain contains textual recordings ('unable', 'asleep', '3-4') that a
#      blind numeric conversion would silently destroy - they are clinical
#      information, not noise. Likewise, a temperature of 34-39 cannot be
#      Fahrenheit (it would be lethal) but is a perfectly normal Celsius
#      body temperature: discarding it as garbage would lose a real
#      measurement. Hypotheses are tested, not assumed.
# Expected: a few hundred textual pain values; a handful of Celsius
#      temperatures convertible to Fahrenheit.

# The seven physiological measures present in the log
measures = ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain']

# Count the RECORDED values first, before any conversion: this is the true
# "measured" denominator for each measure
measured_counts = {}
for col in measures:
    measured_counts[col] = event_logs[col].notna().sum()
    event_logs[col + '_measured'] = event_logs[col].notna()

# Pain is recorded as free text in some cases: look at what we would lose
# with a blind numeric conversion
pain_numeric = pd.to_numeric(event_logs['pain'], errors='coerce')
pain_textual = event_logs['pain'][pain_numeric.isna() & event_logs['pain'].notna()]
print('Pain values recorded: {}'.format(measured_counts['pain']))
print('Pain values that are textual (not convertible to a number): {}'.format(pain_textual.shape[0]))
print('Examples of textual pain values:', pain_textual.unique()[:8].tolist())

# Keep the textual pain recordings documented in a dedicated flag: they mean
# "pain assessment attempted but not scored" (asleep, unable, intubated...)
event_logs['pain_textual'] = pain_numeric.isna() & event_logs['pain'].notna()

# Now convert every measure to numeric (textual values become NaN, but the
# *_measured flags above already preserved the true recorded counts)
for col in measures:
    event_logs[col] = pd.to_numeric(event_logs[col], errors='coerce')

# Unit-error hypothesis for temperature: values far below the Fahrenheit
# range should be Celsius recordings. Test it by looking at them.
# NOTE: the project convention is to keep ALL temperatures in degrees
# Fahrenheit and to state the measurement unit explicitly everywhere.
suspect_temps = event_logs['temperature'][(event_logs['temperature'].notna()) &
                                          (event_logs['temperature'] < 80)]
print('\nTemperatures below 80 °F (impossible in Fahrenheit): {}'.format(sorted(suspect_temps.tolist())))

# All suspect values fall in the 30-45 band, i.e. the normal Celsius body
# temperature range (°C): the unit-error hypothesis is confirmed. We convert
# them to Fahrenheit (°F) with an explicit flag - never silently.
event_logs['temperature_unit_converted'] = (event_logs['temperature'].notna()) & (event_logs['temperature'] < 45)
event_logs.loc[event_logs['temperature_unit_converted'], 'temperature'] = \
    event_logs.loc[event_logs['temperature_unit_converted'], 'temperature'] * 9 / 5 + 32
converted = event_logs[event_logs['temperature_unit_converted']]['temperature']
print('Temperatures converted from Celsius (°C) to Fahrenheit (°F): {}'.format(converted.shape[0]))
print('Converted values (°F): {}'.format(sorted(converted.round(1).tolist())))

# Interpretation: 587 pain recordings are textual clinical notes and are
# preserved in the pain_textual flag instead of being silently dropped; the
# 7 impossible temperatures are Celsius recordings (34.4-38.8 °C) and after
# the declared conversion to °F one of them (38.8 °C = 101.8 °F) turns out
# to be a real fever that a blind cleaning would have thrown away. From
# here on, every temperature in this project is expressed in °F.

In [ ]:
# [NB01-C11] PLAUSIBILITY AND ABNORMALITY FLAGS (APPENDIX, FAHRENHEIT)
# What: flag the values that remain physically implausible after the unit
#       fix, and the values that are clinically abnormal according to the
#       Appendix of the case study - including rhythm.
# Why: implausible values are recording errors and must not feed clinical
#      analyses; abnormal values are real clinical states and become binary
#      patterns in NB04. Keeping the two concepts separate is the core of
#      this audit. Acuity (also in the Appendix) is a case-level attribute
#      and is flagged in the case table below.
# Expected: very few implausible values per measure; substantial abnormal
#      counts, always with their measured denominator.

# Measurement unit of each measure: stated once here and carried into every
# table and export (project convention: units always explicit, temp in °F)
measure_units = {'temperature': '°F',
                 'heartrate':   'bpm',
                 'resprate':    'breaths/min',
                 'o2sat':       '%',
                 'sbp':         'mmHg',
                 'dbp':         'mmHg',
                 'pain':        '0-10 scale'}

# Physically plausible ranges (wide bounds: outside = recording error).
# The temperature upper bound is 116 °F because the highest documented human
# body temperature is 115.7 °F (heatstroke survivor, Atlanta 1980): values
# above that cannot be a live patient.
plausible_ranges = {'temperature': (80, 116),   # °F
                    'heartrate':   (20, 250),   # bpm
                    'resprate':    (5, 80),     # breaths/min
                    'o2sat':       (50, 100),   # %
                    'sbp':         (50, 300),   # mmHg
                    'dbp':         (20, 200),   # mmHg
                    'pain':        (0, 10)}     # 0-10 scale

# Build the plausibility flags measure by measure
for col in measures:
    lo, hi = plausible_ranges[col]
    # implausible = present but outside the physically possible range
    event_logs[col + '_implausible'] = event_logs[col].notna() & ~event_logs[col].between(lo, hi)
    # valid = the value itself, kept only when plausible (else NaN)
    event_logs[col + '_valid'] = event_logs[col].where(event_logs[col].between(lo, hi))

# Abnormal flags computed ONLY on plausible values, thresholds from the Appendix
event_logs['temperature_abnormal'] = event_logs['temperature_valid'] >= 100   # fever >= 100 °F
event_logs['heartrate_abnormal'] = event_logs['heartrate_valid'] > 100        # tachycardia > 100 bpm
event_logs['resprate_abnormal'] = event_logs['resprate_valid'] > 20           # tachypnea > 20 breaths/min
event_logs['o2sat_abnormal'] = event_logs['o2sat_valid'] < 90                 # hypoxemia < 90 %
event_logs['sbp_abnormal'] = event_logs['sbp_valid'] > 120                    # high systolic BP > 120 mmHg
event_logs['dbp_abnormal'] = event_logs['dbp_valid'] > 80                     # high diastolic BP > 80 mmHg
event_logs['pain_abnormal'] = event_logs['pain_valid'] > 0                    # any pain > 0 (0-10 scale)

# Rhythm is categorical: abnormal = a recorded rhythm different from sinus
event_logs['rhythm_measured'] = event_logs['rhythm'].notna()
event_logs['rhythm_abnormal'] = event_logs['rhythm'].notna() & \
    (event_logs['rhythm'].astype(str).str.strip().str.lower() != 'sinus rhythm')

# Print a compact audit table: unit / measured / implausible / abnormal per
# measure (the unit column makes the measurement unit explicit everywhere)
audit_rows = []
for col in measures:
    audit_rows.append({'measure': col,
                       'unit': measure_units[col],
                       'measured': int(measured_counts[col]),
                       'implausible': int(event_logs[col + '_implausible'].sum()),
                       'abnormal': int(event_logs[col + '_abnormal'].sum())})
audit_rows.append({'measure': 'rhythm',
                   'unit': 'categorical',
                   'measured': int(event_logs['rhythm_measured'].sum()),
                   'implausible': 0,
                   'abnormal': int(event_logs['rhythm_abnormal'].sum())})
audit_df = pd.DataFrame(audit_rows)
audit_df

# Interpretation: after the unit fix the residual implausible values are
# rare recording errors (e.g. o2sat 198, dbp 790); they are excluded from
# the *_valid columns and will never feed the clinical patterns. Abnormal
# counts use the Appendix thresholds and always refer to the measured
# (plausible) denominator, which we report alongside.

In [ ]:
# [NB01-C12] DATA QUALITY SUMMARY EXPORT
# What: merge the missingness table and the measurement audit into a single
#       auditable CSV used by the report.
# Why: the report must reference a stable artifact for every data-quality
#      claim; exporting it makes the audit reproducible.
# Expected: data_quality_summary.csv written next to this notebook, with
#      the measured count filled for every column and the implausible /
#      abnormal counts only where they apply (clinical measures).

# Start from the missingness table computed in [NB01-C08]
quality = missing_df.copy()

# measured = number of really recorded values, valid for EVERY column.
# For the physiological measures we use the counts taken BEFORE the numeric
# conversion (textual pain values are recorded values!); for all the other
# columns a plain non-null count is exact.
measured_list = []
for col_name in quality['column']:
    if col_name in measured_counts:
        measured_list.append(int(measured_counts[col_name]))
    else:
        measured_list.append(int(event_logs[col_name].notna().sum()))
quality['measured'] = measured_list

# Attach unit, implausible and abnormal counts where they apply; other
# columns keep an empty value (not a misleading zero)
quality = quality.merge(audit_df.rename(columns={'measure': 'column'})[['column', 'unit', 'implausible', 'abnormal']],
                        on='column', how='left')

# Save the table as CSV for the report (empty cells where not applicable)
quality.to_csv('data_quality_summary.csv', index=False)
print('Saved data_quality_summary.csv with {} rows'.format(len(quality)))

quality

In [ ]:
# [NB01-C13] CASE TABLE (ONE ROW PER STAY)
# What: build a table with one row per stay, holding the case-level
#       attributes, the basic size/duration measures and two case-level
#       flags: urgent triage (Appendix) and clinically interrupted pathway.
# Why: attributes like gender or disposition are constant within a stay but
#      recorded only on some events; taking the first non-null value per
#      case collects them without inventing data. This table feeds the
#      group comparisons of NB02 and the context patterns of NB04.
# Expected: 1,820 rows, one per stay.

# Helper: return the first non-null value of a column inside one case
def first_non_null(values):
    values = values.dropna()
    if len(values) > 0:
        return values.iloc[0]
    return np.nan

# Group the log by case
grouped = event_logs.groupby('case:concept:name')

# Case-level attributes to collect (constant within a stay)
case_attributes = ['gender', 'race', 'arrival_transport', 'disposition', 'acuity', 'chiefcomplaint']

# Build the case table attribute by attribute
case_table = pd.DataFrame(index=grouped.size().index)
for col in case_attributes:
    case_table[col] = grouped[col].agg(first_non_null)

# Number of distinct diagnosis codes recorded for the stay
def count_unique_non_null(values):
    return values.dropna().nunique()
case_table['diagnosis_count'] = grouped['diagnosis_code'].agg(count_unique_non_null)

# Size and duration of the stay
case_table['event_count'] = grouped.size()                        # events in the stay
case_table['start_time'] = grouped['time:timestamp'].min()        # first event
case_table['end_time'] = grouped['time:timestamp'].max()          # last event
case_table['lead_time_min'] = (case_table['end_time'] - case_table['start_time']).dt.total_seconds() / 60

# Appendix flag at case level: acuity 1-3 marks the urgent patients
case_table['acuity_1_3'] = case_table['acuity'].isin([1.0, 2.0, 3.0])

# Clinically interrupted pathways: administratively complete stays whose
# outcome means the treatment did not follow the intended course. They are
# kept and flagged (they are interesting variants), never removed.
interrupted_outcomes = ['LEFT WITHOUT BEING SEEN', 'ELOPED', 'LEFT AGAINST MEDICAL ADVICE', 'EXPIRED', 'OTHER']
case_table['interrupted_pathway'] = case_table['disposition'].isin(interrupted_outcomes)

# Verify the size: one row per stay
print('Rows in the case table: {}'.format(len(case_table)))

# Acuity is missing for some stays: count them and declare the decision -
# these stays are kept, and simply excluded from acuity-based groupings
# (the denominator will always be reported)
print('Stays without a recorded acuity: {}'.format(case_table['acuity'].isna().sum()))

# Count the interrupted pathways
print('Stays flagged as interrupted pathway: {}'.format(case_table['interrupted_pathway'].sum()))

# Quick look at the categorical attributes we will use for group comparisons
print('\nDisposition:\n', case_table['disposition'].value_counts(dropna=False))
print('\nAcuity:\n', case_table['acuity'].value_counts(dropna=False).sort_index())
print('\nArrival transport:\n', case_table['arrival_transport'].value_counts(dropna=False))

case_table.head(10)

# Interpretation: every stay now has its administrative profile in one row.
# 56 stays have no recorded acuity (kept; excluded only from acuity-based
# groups) and 83 stays are interrupted pathways (kept and flagged as their
# own segment). Missing values that remain here are genuinely absent from
# the source data and are reported as such, never imputed.

In [ ]:
# [NB01-C14] STAY DURATION (LEAD TIME) DISTRIBUTION
# What: describe the distribution of the stay duration and test it for
#       normality before choosing which statistics to report.
# Why: on skewed data the mean is misleading (lesson from the course:
#      prefer median and percentiles). The normality test makes the choice
#      explicit instead of assumed.
# Expected: right-skewed distribution, normality clearly rejected.

# Descriptive statistics of the lead time in minutes
print('Mean lead time (min): {:.1f}'.format(case_table['lead_time_min'].mean()))
print('Median lead time (min): {:.1f}'.format(case_table['lead_time_min'].median()))
print('IQR (min): {:.1f} - {:.1f}'.format(case_table['lead_time_min'].quantile(0.25),
                                          case_table['lead_time_min'].quantile(0.75)))
print('p90 (min): {:.1f}'.format(case_table['lead_time_min'].quantile(0.90)))
print('p95 (min): {:.1f}'.format(case_table['lead_time_min'].quantile(0.95)))
print('Max (min): {:.1f}'.format(case_table['lead_time_min'].max()))

# Plot the histogram of the stay duration in hours (easier to read)
plt.figure(figsize=(10, 5))
plt.hist(case_table['lead_time_min'] / 60, bins=45, edgecolor='black', alpha=0.7)
plt.axvline(case_table['lead_time_min'].median() / 60, color='red', linestyle='--',
            label='median = {:.1f} h'.format(case_table['lead_time_min'].median() / 60))
plt.xlabel('Stay duration (hours)')
plt.ylabel('Number of stays')
plt.title('Distribution of ED stay duration')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

# Save the figure for the report, then show it
plt.savefig('figures/NB01_stay_duration_hist.png', dpi=200)
plt.show()

# Perform the Shapiro-Wilk normality test on the durations
stat, p_value = shapiro(case_table['lead_time_min'])
print('Shapiro-Wilk Test Statistic: {:.4f}, p-value: {:.4f}'.format(stat, p_value))
if p_value > 0.05:
    print('The data likely follows a normal distribution.')
else:
    print('The data does NOT follow a normal distribution.')

# Interpretation: the distribution is right-skewed, with a long tail of
# very long stays, and normality is rejected. All duration comparisons in
# the next notebooks will therefore use median / percentiles and
# non-parametric tests (Mann-Whitney, Kruskal-Wallis), not the mean and
# t-tests; the right tail is kept because it concentrates the
# operationally critical cases.

In [ ]:
# [NB01-C15] COMPLETENESS CHECK OF THE CASES
# What: verify case by case that every stay starts with "Enter the ED",
#       ends with "Discharge from the ED", and has no impossible timestamps.
# Why: the scenario claims completeness, but the course method is to test
#      hypotheses on the log, never to assume them. Only cases that fail
#      these checks would be excluded (with documentation).
# Expected: zero violations, so no case is removed.

# First activity of each case (log is already sorted by case and time)
first_activity = grouped['concept:name'].first()

# Last activity of each case
last_activity = grouped['concept:name'].last()

# Count the cases that do not start / do not end as expected
bad_start = (first_activity != 'Enter the ED').sum()
bad_end = (last_activity != 'Discharge from the ED').sum()
print('Cases not starting with "Enter the ED": {}'.format(bad_start))
print('Cases not ending with "Discharge from the ED": {}'.format(bad_end))

# Check for stays with non-positive duration (end before start is impossible)
non_positive = (case_table['lead_time_min'] <= 0).sum()
print('Cases with non-positive duration: {}'.format(non_positive))

# Administrative completeness is not clinical completeness: the 83 stays
# flagged in [NB01-C13] (left without being seen, eloped, left against
# medical advice, expired, other) are complete in the log but clinically
# interrupted. They stay in the dataset as a flagged segment.
print('Clinically interrupted stays (kept, flagged): {}'.format(case_table['interrupted_pathway'].sum()))

# Interpretation: with zero violations on all three structural checks the
# log is complete and temporally consistent, so the full set of 1,820
# stays moves on to the analysis notebooks; nothing is filtered out at
# this stage. The interrupted pathways are a clinical segment to analyse,
# not a data-quality problem to remove.

In [ ]:
# [NB01-C16] ABSTRACT PROCESS VIEW
# What: build a second view of the log where rows sharing the same
#       (case, timestamp, activity) are collapsed into one row with a
#       counter of how many recordings it aggregates.
# Why: discovery and variant analysis look at the control flow; thousands
#      of repeated clinical recordings at the same minute would create
#      artificial loops in the models. The raw view (kept untouched) stays
#      the source of truth for all clinical analyses.
# Expected: the abstract view has fewer rows than the raw view; the
#      difference equals the surplus rows counted in [NB01-C09].

# Collapse repeated (case, timestamp, activity) rows and count them
abstract_log = event_logs.groupby(['case:concept:name', 'time:timestamp', 'concept:name'], as_index=False).agg(
    events_aggregated=('source_row', 'size'),   # how many raw rows were merged
    source_row=('source_row', 'min'))           # keep the first row id as tie-breaker

# Restore the correct event order inside each case
abstract_log = abstract_log.sort_values(['case:concept:name', 'time:timestamp', 'source_row']).reset_index(drop=True)

# Compare the size of the two views
print('Raw view rows: {}'.format(len(event_logs)))
print('Abstract view rows: {}'.format(len(abstract_log)))
print('Rows aggregated away: {}'.format(len(event_logs) - len(abstract_log)))

# Distribution of the aggregation counter (how many rows were merged per event)
print('\nEvents aggregated per abstract row:\n', abstract_log['events_aggregated'].value_counts().sort_index())

abstract_log.head(10)

# Interpretation: the difference between the two views (25,115 - 16,826 =
# 8,289) matches exactly the surplus rows found in [NB01-C09], confirming
# the aggregation did exactly what was intended and nothing more.
# NB02/NB03 will run control-flow analyses on both views as a sensitivity
# check.

In [ ]:
# [NB01-C17] EXPORT OF THE PREPARED FILES
# What: save the prepared raw log (with quality flags), the abstract view
#       and the case table as CSV files.
# Why: the next notebooks (NB02-NB05) reload these files in their header
#      instead of repeating the cleaning, keeping every notebook runnable
#      on its own (same pattern as the course notebooks of Case8).
# Expected: three CSV files written next to this notebook.

# Save the prepared raw event log (all columns, flags included)
event_logs.to_csv('prepared_event_log.csv', index=False)
print('Saved prepared_event_log.csv ({} rows)'.format(len(event_logs)))

# Save the abstract control-flow view
abstract_log.to_csv('abstract_event_log.csv', index=False)
print('Saved abstract_event_log.csv ({} rows)'.format(len(abstract_log)))

# Save the case table (index is the case id, so we keep it)
case_table.to_csv('case_table.csv')
print('Saved case_table.csv ({} rows)'.format(len(case_table)))

# Final summary of what NB01 produced
print('\nNB01 outputs: prepared_event_log.csv, abstract_event_log.csv,')
print('case_table.csv, data_quality_summary.csv, 3 figures in figures/')